In [135]:
import os
os.environ["TOGETHER_API_KEY"] = "tgp_v1_--nKOPx_Rtr2sAiSOtvpafrbcvAjyajARRfAyZ8G_Bo"

In [136]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [137]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [138]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [5]:
multiply.name

'multiply'

In [6]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [7]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [ ]:
# tool binding

In [139]:
from langchain_together import  ChatTogether

In [140]:
llm = ChatTogether(
        together_api_key=os.environ["TOGETHER_API_KEY"],
        model="Qwen/Qwen3.5-9B",
        temperature=0.3,
        max_tokens=512
    )

In [141]:
llm.invoke('hi')

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 512, 'prompt_tokens': 11, 'total_tokens': 523, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_name': 'Qwen/Qwen3.5-9B', 'system_fingerprint': None, 'id': 'ohghKmb-6Ng1vN-9f643a494d5ac6bc', 'service_tier': None, 'finish_reason': 'length', 'logprobs': None}, id='run--019df0e9-a8a3-70a3-a7ba-6d1e39644f75-0', usage_metadata={'input_tokens': 11, 'output_tokens': 512, 'total_tokens': 523, 'input_token_details': {}, 'output_token_details': {}})

In [142]:
llm_with_tools = llm.bind_tools([multiply])

In [143]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hello! I'm doing well, thank you for asking! How about you? Is there anything I can help you with today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 293, 'total_tokens': 356, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_name': 'Qwen/Qwen3.5-9B', 'system_fingerprint': None, 'id': 'ohghrKz-7ArivV-9f643ccc3c022698', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--019df0eb-3a7f-7673-89f7-48fb6186d6f3-0', usage_metadata={'input_tokens': 293, 'output_tokens': 63, 'total_tokens': 356, 'input_token_details': {}, 'output_token_details': {}})

In [144]:
query = HumanMessage('can you multiply 3 with 1000')

In [145]:
messages = [query]

In [146]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [147]:
result = llm_with_tools.invoke(messages)

In [148]:
messages.append(result)

In [149]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_8f97d88b4eaf4e01a907c1c3', 'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 300, 'total_tokens': 387, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_name': 'Qwen/Qwen3.5-9B', 'system_fingerprint': None, 'id': 'ohgiHc2-7ArivV-9f643ecddb677643', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019df0ec-7a87-70c0-afb6-cac84c9dd33b-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_8f97d88b4eaf4e01a907c1c3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 300, 'output_tokens': 87, 'total_tokens': 387, 'input_token_details': {}, 'output_token_details': {}})]

In [150]:
if result.tool_calls and len(result.tool_calls) > 0:
    tool_result = multiply.invoke(result.tool_calls[0])
else:
    print("No tool calls were made by the model")

In [151]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_8f97d88b4eaf4e01a907c1c3')

In [152]:
messages.append(tool_result)

In [153]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_8f97d88b4eaf4e01a907c1c3', 'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 300, 'total_tokens': 387, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_name': 'Qwen/Qwen3.5-9B', 'system_fingerprint': None, 'id': 'ohgiHc2-7ArivV-9f643ecddb677643', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019df0ec-7a87-70c0-afb6-cac84c9dd33b-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_8f97d88b4eaf4e01a907c1c3', 'type': 'tool_call'}], usage_metadata={'input_tokens': 300, 'output_tokens': 87, 'total_tokens': 387, 'input_token_details': {}, 'output_token_details': {}}),
 Tool

In [154]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [155]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [156]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1777852801,
 'time_last_update_utc': 'Mon, 04 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1777939201,
 'time_next_update_utc': 'Tue, 05 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.0964}

In [157]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [158]:
# tool binding
llm = ChatTogether(
        together_api_key=os.environ["TOGETHER_API_KEY"],
        model="Qwen/Qwen3.5-9B",
        temperature=0.3,
        max_tokens=512
    )

In [159]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [169]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10000 usd to inr')]

In [170]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10000 usd to inr', additional_kwargs={}, response_metadata={})]

In [171]:
ai_message = llm_with_tools.invoke(messages)

In [172]:
messages.append(ai_message)

In [173]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_a19c2a640222490798ce7d5a',
  'type': 'tool_call'}]

In [174]:
import json

# Process all tool calls in a loop until no more tool calls are made
while True:
    tool_calls = ai_message.tool_calls if hasattr(ai_message, 'tool_calls') and ai_message.tool_calls else []
    
    if not tool_calls:
        break
    
    for tool_call in tool_calls:
        # execute the 1st tool and get the value of conversion rate
        if tool_call['name'] == 'get_conversion_factor':
            tool_message1 = get_conversion_factor.invoke(tool_call)
            # fetch this conversion rate
            conversion_rate = json.loads(tool_message1.content)['conversion_rate']
            # append this tool message to messages list
            messages.append(tool_message1)
        # execute the 2nd tool using the conversion rate from tool 1
        if tool_call['name'] == 'convert':
            # fetch the current arg
            tool_call['args']['conversion_rate'] = conversion_rate
            tool_message2 = convert.invoke(tool_call)
            messages.append(tool_message2)
            print(f"Final Converted Value: {tool_message2}")
    
    # Invoke LLM again to get next tool calls (if any)
    ai_message = llm_with_tools.invoke(messages)
    messages.append(ai_message)


Final Converted Value: content='950964.0' name='convert' tool_call_id='call_e1b9d8e842084de49362ecf8'


In [167]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10000 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_9d71058c918c4e23995cfe81', 'function': {'arguments': '{"base_currency": "INR", "target_currency": "USD"}', 'name': 'get_conversion_factor'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 405, 'total_tokens': 549, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_name': 'Qwen/Qwen3.5-9B', 'system_fingerprint': None, 'id': 'ohgkzmr-3pDw3Z-9f644bc25fc590f3', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019df0f4-9439-7ff1-b341-42559f4ad60c-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'call_9d71058c918c4e23995cfe81', 't

In [168]:
# --- Step 6: Run the Agent ---
user_query = "how much is 100 USD in INR?"

# Use the manual approach with LLM to get tool calls and execute them
agent_messages = [HumanMessage(user_query)]

# First call to get initial tool calls
ai_resp = llm_with_tools.invoke(agent_messages)
agent_messages.append(ai_resp)

print("=" * 50)
print(f"User Query: {user_query}")
print("=" * 50)

# Process tool calls in a loop until no more tool calls
while True:
    tool_calls = ai_resp.tool_calls if hasattr(ai_resp, 'tool_calls') and ai_resp.tool_calls else []
    
    if not tool_calls:
        # No tool calls, get the final answer
        if hasattr(ai_resp, 'content') and ai_resp.content:
            print(f"Final Answer: {ai_resp.content}")
        break
    
    # Execute tool calls
    for tool_call in tool_calls:
        if tool_call['name'] == 'get_conversion_factor':
            tool_message = get_conversion_factor.invoke(tool_call)
            conversion_rate = json.loads(tool_message.content)['conversion_rate']
            agent_messages.append(tool_message)
            print(f"Conversion Rate: {conversion_rate}")
            
        elif tool_call['name'] == 'convert':
            # Add the conversion rate to tool call args
            tool_call['args']['conversion_rate'] = conversion_rate
            tool_message = convert.invoke(tool_call)
            agent_messages.append(tool_message)
            converted_value = tool_message.content
            print(f"Converted Value: {converted_value}")
    
    # Get next response from LLM
    ai_resp = llm_with_tools.invoke(agent_messages)
    agent_messages.append(ai_resp)

print("=" * 50)


User Query: how much is 100 USD in INR?
Conversion Rate: 95.0964
Converted Value: 9509.64
Final Answer: 100 USD is approximately 9,509.64 INR.
